In [1]:
wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv

SyntaxError: invalid syntax (3069519253.py, line 1)

In [37]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
!pip install xgboost
import xgboost as xgb

import matplotlib.pyplot as plt

You should consider upgrading via the '/Users/hiteshallakki/Documents/DataTalks/jupyter_env/bin/python3 -m pip install --upgrade pip' command.


In [23]:
import pandas as pd

url = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv'
df = pd.read_csv(url)
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [24]:
df.columns = df.columns.str.lower()
df.columns

Index(['engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'model_year', 'origin', 'fuel_type', 'drivetrain',
       'num_doors', 'fuel_efficiency_mpg'],
      dtype='object')

In [25]:
df.describe().round(2)

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,num_doors,fuel_efficiency_mpg
count,9704.00,9222.00,8996.00,9704.00,8774.00,9704.00,9202.00,9704.00
mean,199.71,3.96,149.66,3001.28,15.02,2011.48,-0.01,14.99
std,49.46,2.00,29.88,497.89,2.51,6.66,1.05,2.56
min,10.00,0.00,37.00,952.68,6.00,2000.00,-4.00,6.20
25%,170.00,3.00,130.00,2666.25,13.30,2006.00,-1.00,13.27
50%,200.00,4.00,149.00,2993.23,15.00,2012.00,0.00,15.01
75%,230.00,5.00,170.00,3334.96,16.70,2017.00,1.00,16.71
max,380.00,13.00,271.00,4739.08,24.30,2023.00,4.00,25.97


In [26]:
df.num_doors.value_counts()

num_doors
 0.0    3551
 1.0    2192
-1.0    2183
-2.0     594
 2.0     563
 3.0      58
-3.0      56
-4.0       4
 4.0       1
Name: count, dtype: int64

In [11]:
df.columns

Index(['engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'model_year', 'origin', 'fuel_type', 'drivetrain',
       'num_doors', 'fuel_efficiency_mpg'],
      dtype='object')

In [33]:
df = df.fillna(0)

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369
...,...,...,...,...,...,...,...,...,...,...,...
9699,140,5.0,164.0,2981.107371,17.3,2013,Europe,Diesel,Front-wheel drive,NaN,15.101802
9700,180,NaN,154.0,2439.525729,15.0,2004,USA,Gasoline,All-wheel drive,0.0,17.962326
9701,220,2.0,138.0,2583.471318,15.1,2008,USA,Diesel,All-wheel drive,-1.0,17.186587
9702,230,4.0,177.0,2905.527390,19.4,2011,USA,Diesel,Front-wheel drive,1.0,15.331551


In [38]:
df_full_train,df_test = train_test_split(df,test_size = 0.2, random_state =1)
df_train,df_val = train_test_split(df_full_train , test_size = 0.25,random_state =1)

In [39]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

In [40]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [41]:
numerical = ['engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight', 'acceleration', 'model_year']

In [42]:
categorical = ['origin', 'fuel_type', 'drivetrain', 'num_doors']

In [43]:
features = numerical + categorical 

In [44]:

#dictVectorizer
dv = DictVectorizer(sparse = True)
train_dicts = df_train[features].to_dict(orient = 'records')
X_train = dv.fit_transform(train_dicts)
X_train

<5822x14 sparse matrix of type '<class 'numpy.float64'>'
	with 58220 stored elements in Compressed Sparse Row format>

In [47]:
val_dicts = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [48]:
test_dicts = df_test[features].to_dict(orient='records')
X_test = dv.transform(test_dicts)

Question 1 : Decision Tree Regressor(max_depth =1)

In [55]:
dt = DecisionTreeRegressor(max_depth = 1 , random_state = 1)
dt.fit(X_train, y_train)

# we look at feature_importances
feature_names = dv.get_feature_names_out()
importance = pd.Series(dt.feature_importances_,index = feature_names)
split_feature = importance.sort_values(ascending = False).index[0]
print(split_feature)

vehicle_weight


Question 2 : Random Forest Regressor RMSE
n_estimators = 10
random_state  = 1
n_jobs = 1

In [59]:
rf = RandomForestRegressor(n_estimators = 10 , random_state = 1, n_jobs =-1)
rf.fit(X_train , y_train)

y_pred_val = rf.predict(X_val)
rmse  = np.sqrt(mean_squared_error (y_val,y_pred_val))
rmse

np.float64(0.4586615458484907)

Question 3 Question 3: Tuning n_estimators
Experiment with n_estimators from 10 to 200 with a step of 10 ([10, 20, ..., 200]). Set random_state=1. Evaluate the model on the validation dataset (RMSE). After which value of n_estimators does RMSE stop improving?

In [60]:
scores = []
for n in range(10,201,10):
    rf = RandomForestRegressor(n_estimators = n , random_state = 1, n_jobs =-1)
    rf.fit(X_train , y_train)
    y_pred_val = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val,y_pred_val))
    scores.append((n,rmse))
df_scores = pd.DataFrame(scores, columns=['n_estimators', 'rmse'])


In [61]:
df_scores

,n_estimators,rmse
0,10,0.458662
1,20,0.453680
2,30,0.451172
3,40,0.448357
4,50,0.446179
5,60,0.445300
6,70,0.444674
7,80,0.444994
8,90,0.445205
9,100,0.444896


Question 4
Try different values of max_depth: [10, 15, 20, 25]. For each max_depth, try n_estimators from 10 to 200 (step 10). Calculate the mean RMSE across all n_estimators for a given max_depth. Fix the random seed: random_state=1.

In [66]:
depths = [10,15,20,25]
mean_rmse = {}
for d in depths:
    scores = []
    for n in range(10,201,10):
        rf = RandomForestRegressor(n_estimators = n , max_depth = d, random_state = 1 , n_jobs = -1)
        rf.fit(X_train,y_train)
        y_pred_val = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val,y_pred_val))
        scores.append(rmse)
    mean_rmse[d] = np.mean(scores)

print(mean_rmse)

{10: np.float64(0.4418792992525217), 15: np.float64(0.44561628816456206), 20: np.float64(0.4456793443309614), 25: np.float64(0.44570249863475137)}


In [72]:
mean_rmse

{10: np.float64(0.4418792992525217),
 15: np.float64(0.44561628816456206),
 20: np.float64(0.4456793443309614),
 25: np.float64(0.44570249863475137)}

question 5 Question 5: Feature Importance
Train a Random Forest model with:

n_estimators=10

max_depth=20

random_state=1

n_jobs=-1

Find the most important feature using the feature_importances_ attribute.

In [73]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()
importance = pd.Series(rf.feature_importances_, index=feature_names)
most_important_feature = importance.sort_values(ascending=False).index[0]

In [75]:
most_important_feature

'vehicle_weight'

Question 6 
Question 6: XGBoost eta Tuning
Train an XGBoost model for 100 rounds with two different eta values: 0.3 and 0.1. Compare the RMSE score on the validation dataset.

In [79]:
feature_name_list = dv.get_feature_names_out().tolist()

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_name_list)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_name_list)
watchlist = [(dtrain, 'train') ,(dval ,'val')]

xgb_base_params = {
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 0, # Set to 0 for cleaner output
}
num_round = 100

In [83]:
# 1. Define the dictionary to store results
evals_result = {} 

params_03 = xgb_base_params.copy()
params_03['eta'] = 0.3

model_03 = xgb.train(
    params_03, 
    dtrain, 
    num_round, 
    watchlist,
    evals_result=evals_result, 
    verbose_eval=False
)


val_rmse_03 = evals_result['val']['rmse'][-1]
print(f"RMSE (eta=0.3): {val_rmse_03:.4f}")

RMSE (eta=0.3): 0.4502


In [ ]:
# 1. Define the dictionary to store results
evals_result = {} 

params_01 = xgb_base_params.copy()
params_01['eta'] = 0.1

model_01 = xgb.train(
    params_01, 
    dtrain, 
    num_round, 
    watchlist,
    evals_result=evals_result, 
    verbose_eval=False
)


val_rmse_03 = evals_result['val']['rmse'][-1]
print(f"RMSE (eta=0.3): {val_rmse_03:.4f}")